# 3. Modelos de Ensemble & Feature Engineering Avançado
**Unidade Curricular:** Dados e Aprendizagem Automática (DAA)  
**Objetivo:** Implementar técnicas de Feature Engineering no dataset do Inquérito Nacional de Saúde, treinar e comparar modelos de Ensemble (Random Forest, LightGBM, XGBoost, CatBoost) e aplicar Validação Cruzada Estratificada ($10$-fold CV) para otimizar o F1-Score (Macro).

In [14]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Configurações gerais
sns.set_theme(style="whitegrid")

# Caminhos dos ficheiros
DATA_DIR = "../data"
TRAIN_CLEAN_PATH = os.path.join(DATA_DIR, "train_clean.csv")
TEST_CLEAN_PATH = os.path.join(DATA_DIR, "test_clean.csv")

# 1. Carregar os dados limpos
train_df = pd.read_csv(TRAIN_CLEAN_PATH)
test_df = pd.read_csv(TEST_CLEAN_PATH)

target_col = "AGE_RANGE"

print(f"Dimensões Iniciais do Treino: {train_df.shape}")
print(f"Dimensões Iniciais do Teste: {test_df.shape}")

Dimensões Iniciais do Treino: (10962, 235)
Dimensões Iniciais do Teste: (3655, 234)


## 3.1 Feature Engineering Avançado
Criação de novas variáveis explicativas com base em relações sociodemográficas, indicadores de saúde agregados e padrões de não-resposta no inquérito.

In [15]:
def apply_feature_engineering(df):
    df_feat = df.copy()

    # 1. Rácio de Membros Ativos / Inativos na Casa
    if "HH_ACT" in df_feat.columns and "HH_INACT" in df_feat.columns:
        df_feat["HH_ACTIVE_RATIO"] = df_feat["HH_ACT"] / (
            df_feat["HH_ACT"] + df_feat["HH_INACT"] + 1
        )
        df_feat["HH_TOTAL_MEMBERS"] = df_feat["HH_ACT"] + df_feat["HH_INACT"]

    # 2. Contagem de Respostas Não Aplicáveis / Omisso (-1) por indivíduo
    df_feat["TOTAL_MISSING_COUNT"] = (df_feat == -1).sum(axis=1)

    # 3. Transformação Logarítmica do Peso Amostral (WGT)
    if "WGT" in df_feat.columns:
        df_feat["WGT_LOG"] = np.log1p(np.maximum(0, df_feat["WGT"]))

    # 4. Agregação de Questões de Saúde / Hábitos (colunas que começam por HA, HS, CD, IN)
    health_cols = [
        c
        for c in df_feat.columns
        if c.startswith(("HA", "HS", "CD", "IN")) and c != "IN16"
    ]
    if len(health_cols) > 0:
        df_feat["HEALTH_ISSUES_SUM"] = (df_feat[health_cols] > 0).sum(axis=1)
        df_feat["HEALTH_ISSUES_MEAN"] = df_feat[health_cols].mean(axis=1)

    return df_feat


# Aplicar transformações nos dois conjuntos
train_fe = apply_feature_engineering(train_df)
test_fe = apply_feature_engineering(test_df)

X_train = train_fe.drop(columns=[target_col])
y_train = train_fe[target_col]
X_test = test_fe.drop(columns=[target_col], errors="ignore")

# Configurar Validação Cruzada Estratificada
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"Novas dimensões de X_train após Feature Engineering: {X_train.shape}")
print(f"Novas dimensões de X_test após Feature Engineering: {X_test.shape}")

Novas dimensões de X_train após Feature Engineering: (10962, 240)
Novas dimensões de X_test após Feature Engineering: (3655, 240)


## 3.2 Random Forest Classifier (Bagging)
Avaliação do modelo Random Forest de referência ($300$ árvores com `class_weight='balanced'`).

In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)

cv_scores_rf = cross_val_score(
    rf_model, X_train, y_train, cv=skf, scoring="f1_macro", n_jobs=-1
)

print("--- RESULTADOS RANDOM FOREST (10-FOLD CV) ---")
print(f"F1-Score (Macro) por fold: {np.round(cv_scores_rf, 4)}")
print(
    f"F1-Score (Macro) Médio: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std():.4f})"
)

--- RESULTADOS RANDOM FOREST (10-FOLD CV) ---
F1-Score (Macro) por fold: [0.4552 0.4631 0.4595 0.4469 0.4391 0.4247 0.4391 0.4594 0.4606 0.4527]
F1-Score (Macro) Médio: 0.4500 (+/- 0.0118)


## 3.3 LightGBM Classifier (Gradient Boosting)
Modelo otimizado de Gradient Boosting baseado em árvores com crescimento focado nas folhas (*leaf-wise*).

In [17]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=16,
    n_estimators=150,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

cv_scores_lgb = cross_val_score(
    lgb_model, X_train, y_train, cv=skf, scoring="f1_macro", n_jobs=-1
)

print("--- RESULTADOS LIGHTGBM (10-FOLD CV) ---")
print(f"F1-Score (Macro) por fold: {np.round(cv_scores_lgb, 4)}")
print(
    f"F1-Score (Macro) Médio: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std():.4f})"
)

--- RESULTADOS LIGHTGBM (10-FOLD CV) ---
F1-Score (Macro) por fold: [0.5995 0.5847 0.5942 0.5699 0.6106 0.5688 0.5801 0.5822 0.5778 0.5553]
F1-Score (Macro) Médio: 0.5823 (+/- 0.0153)


## 3.4 XGBoost Classifier (Extreme Gradient Boosting)
Modelo de Gradient Boosting com regularização L1/L2 integrada para mitigar o sobreajuste.

In [18]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=16,
    n_estimators=150,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
)

cv_scores_xgb = cross_val_score(
    xgb_model, X_train, y_train, cv=skf, scoring="f1_macro", n_jobs=-1
)

print("--- RESULTADOS XGBOOST (10-FOLD CV) ---")
print(f"F1-Score (Macro) por fold: {np.round(cv_scores_xgb, 4)}")
print(
    f"F1-Score (Macro) Médio: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std():.4f})"
)

--- RESULTADOS XGBOOST (10-FOLD CV) ---
F1-Score (Macro) por fold: [0.5667 0.5788 0.575  0.551  0.5808 0.5523 0.55   0.5654 0.5554 0.5315]
F1-Score (Macro) Médio: 0.5607 (+/- 0.0147)


## 3.5 CatBoost Classifier (Opcional - Treino Avançado)
Algoritmo de Gradient Boosting especialmente otimizado para lidar nativamente com variáveis categóricas e ordinais.

In [19]:
from catboost import CatBoostClassifier

cb_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.08,
    depth=6,
    loss_function="MultiClass",
    random_seed=42,
    verbose=0,
    thread_count=-1,
)

cv_scores_cb = cross_val_score(
    cb_model, X_train, y_train, cv=skf, scoring="f1_macro", n_jobs=-1
)

print("--- RESULTADOS CATBOOST (10-FOLD CV) ---")
print(f"F1-Score (Macro) por fold: {np.round(cv_scores_cb, 4)}")
print(
    f"F1-Score (Macro) Médio: {cv_scores_cb.mean():.4f} (+/- {cv_scores_cb.std():.4f})"
)

--- RESULTADOS CATBOOST (10-FOLD CV) ---
F1-Score (Macro) por fold: [0.4857 0.5065 0.4993 0.4942 0.5148 0.4722 0.4415 0.4964 0.4882 0.4827]
F1-Score (Macro) Médio: 0.4881 (+/- 0.0193)


## 3.6 Comparação de Modelos e Seleção do Melhor para Submissão
Seleção automática do modelo com maior pontuação no $10$-fold CV e exportação do ficheiro final para a pasta `submissions/`.

In [20]:
models_dict = {
    "RandomForest": (rf_model, cv_scores_rf.mean()),
    "LightGBM": (lgb_model, cv_scores_lgb.mean()),
    "XGBoost": (xgb_model, cv_scores_xgb.mean()),
    "CatBoost": (cb_model, cv_scores_cb.mean()),
}

best_name = max(models_dict, key=lambda k: models_dict[k][1])
best_model, best_score = models_dict[best_name]

print(f"🏆 MELHOR MODELO: {best_name} com F1-Score (Macro) CV = {best_score:.4f}")

# Treinar o melhor modelo com todos os dados
best_model.fit(X_train, y_train)
best_preds = best_model.predict(X_test)

# Se as previsões forem uma matriz 2D (caso do CatBoost/XGBoost em certos casos), achatar para 1D
if len(best_preds.shape) > 1 and best_preds.shape[1] == 1:
    best_preds = best_preds.ravel()

os.makedirs("../submissions", exist_ok=True)
sub_path = f"../submissions/submission_{best_name.lower()}_fe.csv"

sub_df = pd.DataFrame({"RowId": test_df.index, "AGE_RANGE": best_preds})
sub_df.to_csv(sub_path, index=False)

print(f"Ficheiro guardado com sucesso em: {sub_path}")
print(sub_df.head())

🏆 MELHOR MODELO: LightGBM com F1-Score (Macro) CV = 0.5823
Ficheiro guardado com sucesso em: ../submissions/submission_lightgbm_fe.csv
   RowId  AGE_RANGE
0      0          5
1      1          2
2      2         10
3      3          6
4      4          9
